<a href="https://colab.research.google.com/github/braim/nids-tta/blob/main/colab/7-kan-cnn/NIDS_7_5_0_CTTA_TONS_MeM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NIDS — AE+Classifier with Few-Shot Layer-Selective CTTA

## Design

**Pre-training:** supervised CE + reconstruction MSE on fully labelled source.

**CTTA strategy — Layer-Selective Updates:**
Three parameter groups are updated during CTTA:
1. **Norm layers** (LayerNorm, GroupNorm) — adapt feature normalisation statistics
2. **Classifier head** — directly moves the decision boundary toward target patterns
3. **Last encoder layer** — adapts the final feature representation

The early encoder layers (source feature extractors) remain frozen.
This gives the CE loss a direct path to update the decision boundary
while protecting source representations from catastrophic forgetting.

**Three loss signals per batch (all requiring only 1% labelled pool):**
1. Supervised CE on labelled pool — moves decision boundary to target domain
2. Entropy minimisation on stream — increases prediction confidence
3. Reconstruction on pool — keeps representations grounded

## Architectures
| `ARCH` | Description |
|---|---|
| `'kan'` | KAN encoder/decoder + linear classifier |
| `'cnn'` | CNN encoder/decoder + linear classifier |

## Hyperparameters
| Parameter | Description |
|---|---|
| `RECON_W` | Reconstruction weight during pre-training |
| `FEW_SHOT_RATIO` | Fraction of target data used as labelled pool |
| `FEW_SHOT_W` | Supervised CE weight during CTTA |
| `TTA_LR` | Learning rate for layer-selective updates |
| `TTA_STEPS` | Gradient steps per batch |
| `ENTROPY_W` | Entropy minimisation weight |
| `RECON_W_TTA` | Reconstruction weight during CTTA |

In [ ]:
!pip install -q git+https://github.com/Blealtan/efficient-kan.git
!pip install -q polars kagglehub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Imports & Configuration

In [ ]:
import os, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import polars as pl
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from efficient_kan import KAN

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Architecture ──────────────────────────────────────────────────────────────
ARCH            = 'kan'      # 'kan' | 'cnn'

# ── Data ──────────────────────────────────────────────────────────────────────
SAMPLE_N        = 100_000
BATCH_SIZE      = 128
TTA_BATCH_SIZE  = 512        # larger batches = more stable gradient estimates

# ── Model ─────────────────────────────────────────────────────────────────────
LATENT_DIM      = 32
NUM_SLOTS       = 100
SHRINK_THR      = 0.0025

# ── Pre-training ──────────────────────────────────────────────────────────────
PRETRAIN_EPOCHS = 20
PRETRAIN_LR     = 1e-3
WEIGHT_DECAY    = 1e-4
RECON_W         = 0.5        # reconstruction regularises encoder for transfer

# ── CTTA ──────────────────────────────────────────────────────────────────────
FEW_SHOT_RATIO  = 0.01       # fraction of target used as benign pool
FEW_SHOT_W      = 1.0        # supervised CE weight during CTTA
TTA_LR          = 1e-3       # higher lr is fine — only norm params updated
TTA_STEPS       = 1
ENTROPY_W       = 1.0        # entropy minimisation weight
RECON_W_TTA     = 0.5        # reconstruction on benign pool weight

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[System] Seed={SEED} | Device={device} | Arch={ARCH}')

[System] Seed=42 | Device=cpu | Arch=kan


## 2. Data Loading & Preprocessing

In [ ]:
def engineer_features(df: pl.DataFrame) -> pl.DataFrame:
    """Derive flow-level features and drop identifier/label columns."""
    if 'FLOW_END_MILLISECONDS' in df.columns and 'FLOW_START_MILLISECONDS' in df.columns:
        df = df.with_columns(
            (pl.col('FLOW_END_MILLISECONDS') - pl.col('FLOW_START_MILLISECONDS')).alias('FLOW_DURATION')
        )
    else:
        df = df.with_columns(pl.lit(0).alias('FLOW_DURATION'))
    if 'IN_BYTES' in df.columns and 'IN_PKTS' in df.columns:
        df = df.with_columns(
            (pl.col('IN_BYTES') / (pl.col('IN_PKTS') + 1e-5)).alias('BYTES_PER_PKT')
        )
    log_cols = ['IN_BYTES', 'IN_PKTS', 'FLOW_DURATION', 'SRC_TO_DST_IAT_MAX', 'DST_TO_SRC_IAT_MAX']
    existing = [c for c in log_cols if c in df.columns]
    if existing:
        df = df.with_columns([pl.col(c).log1p() for c in existing])
    drop_cols = [
        'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS',
        'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT',
        'Label', 'Attack', 'label', 'attack', 'Date',
    ]
    df = df.drop([c for c in drop_cols if c in df.columns])
    return df


def load_dataset(dataset_name: str, sample_n: int = None):
    """Download dataset and return (X, y). Uses random sampling."""
    print(f'[Data] Loading {dataset_name} ...')
    path = kagglehub.dataset_download(dataset_name)
    csv_files = [
        os.path.join(root, f)
        for root, _, files in os.walk(path)
        for f in files if f.endswith('.csv')
    ]
    df = pl.scan_csv(csv_files[0]).collect(engine='streaming')
    if sample_n and sample_n < df.height:
        df = df.sample(n=sample_n, seed=SEED)
    label_col = next((c for c in df.columns if c.lower() == 'label'), None)
    y = df[label_col].to_numpy().astype(np.int64) if label_col else np.zeros(df.height, dtype=np.int64)
    df = engineer_features(df)
    X  = df.to_numpy().astype(np.float32)
    X  = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    print(f'   -> Shape: {X.shape} | Attack rate: {np.mean(y):.2%}')
    return X, y


def make_source_loaders(X, y):
    """
    Stratified 80/20 split. Scaler fitted on full training split.
    Training loader contains all labelled samples (benign + attack).
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(X_tr)
    X_tr   = np.clip(scaler.transform(X_tr).astype(np.float32), -1, 1)
    X_te   = np.clip(scaler.transform(X_te).astype(np.float32), -1, 1)
    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    loader_tr = DataLoader(train_ds, batch_size=BATCH_SIZE,     shuffle=True)
    loader_te = DataLoader(test_ds,  batch_size=TTA_BATCH_SIZE, shuffle=False)
    return loader_tr, loader_te, scaler


def make_target_loaders(X, y):
    """
    Fit a fresh MinMaxScaler on the full target dataset (no label leakage).

    Stratified split into:
      pool_loader   : FEW_SHOT_RATIO of data, labelled (both classes preserved)
      stream_loader : remaining data, labels kept for evaluation only

    The pool is used for supervised CE anchoring during CTTA.
    Real-world justification: the pool represents a brief initial analyst
    review period at deployment — a realistic assumption.
    """
    scaler   = MinMaxScaler(feature_range=(-1, 1)).fit(X)
    X_scaled = np.clip(scaler.transform(X).astype(np.float32), -1, 1)

    # Stratified split — both classes represented in pool
    X_pool, X_stream, y_pool, y_stream = train_test_split(
        X_scaled, y,
        test_size=(1 - FEW_SHOT_RATIO),
        random_state=SEED,
        stratify=y,
    )

    pool_ds   = TensorDataset(torch.from_numpy(X_pool),   torch.from_numpy(y_pool))
    stream_ds = TensorDataset(torch.from_numpy(X_stream), torch.from_numpy(y_stream))

    pool_loader   = DataLoader(pool_ds,   batch_size=BATCH_SIZE,     shuffle=True)
    stream_loader = DataLoader(stream_ds, batch_size=TTA_BATCH_SIZE, shuffle=True)

    print(f'   -> Pool: {len(y_pool)} samples '
          f'(attack rate: {np.mean(y_pool):.2%}) | '
          f'Stream: {len(y_stream)} '
          f'(attack rate: {np.mean(y_stream):.2%})')
    return pool_loader, stream_loader


## 3. Model Architectures

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Memory Module  (Gong et al. 2019)
# ─────────────────────────────────────────────────────────────────────────────
class MemoryModule(nn.Module):
    """
    Learnable bank of normal-traffic prototypes.
    Encoder query z attends over NUM_SLOTS slots via cosine similarity.
    Hard shrinkage produces sparse addressing.
    Attacks have no matching prototype — poor reconstruction and
    uncertain classification by construction.
    """
    def __init__(self, num_slots: int, latent_dim: int, shrink_thr: float = 0.0025):
        super().__init__()
        self.shrink_thr = shrink_thr
        self.memory     = nn.Parameter(torch.randn(num_slots, latent_dim))
        nn.init.kaiming_uniform_(self.memory)

    def forward(self, z: torch.Tensor):
        z_n   = F.normalize(z,           dim=1)
        m_n   = F.normalize(self.memory, dim=1)
        attn  = F.softmax(torch.matmul(z_n, m_n.T), dim=1)  # (B, M)
        attn  = F.relu(attn - self.shrink_thr)
        attn  = attn / (attn.sum(dim=1, keepdim=True) + 1e-8)
        z_hat = torch.matmul(attn, self.memory)              # (B, D)
        return z_hat, attn


# ─────────────────────────────────────────────────────────────────────────────
# KAN AE+Classifier with Memory
# ─────────────────────────────────────────────────────────────────────────────
class KanAEClassifier(nn.Module):
    """
    Shared KAN encoder → MemoryModule → classifier head + decoder head.

    Encoder:    input_dim -> 64 -> latent_dim  (KAN)
    Memory:     latent_dim -> z_hat  (sparse prototype addressing)
    Classifier: z_hat -> 2           (Linear)
    Decoder:    z_hat -> 64 -> input_dim  (KAN)

    forward() returns (logits, recon, z_hat)
    """
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder    = KAN([input_dim, 64, latent_dim], grid_range=[-1, 1])
        self.ln         = nn.LayerNorm(latent_dim)
        self.memory     = MemoryModule(NUM_SLOTS, latent_dim, SHRINK_THR)
        self.classifier = nn.Linear(latent_dim, 2)
        self.decoder    = KAN([latent_dim, 64, input_dim], grid_range=[-1, 1])

    def forward(self, x):
        z           = self.ln(self.encoder(x))
        z_hat, _    = self.memory(z)
        return self.classifier(z_hat), self.decoder(z_hat), z_hat


# ─────────────────────────────────────────────────────────────────────────────
# CNN AE+Classifier with Memory
# ─────────────────────────────────────────────────────────────────────────────
class CnnAEClassifier(nn.Module):
    """
    Shared CNN encoder → MemoryModule → classifier head + decoder head.

    Encoder:    input_dim -> 64 -> latent_dim  (Conv1d pointwise)
    Memory:     latent_dim -> z_hat
    Classifier: z_hat -> 2
    Decoder:    z_hat -> 64 -> input_dim  (ConvTranspose1d)

    forward() returns (logits, recon, z_hat)
    """
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=1),
            nn.GroupNorm(1, 64), nn.GELU(),
            nn.Conv1d(64, latent_dim, kernel_size=1),
        )
        self.ln         = nn.LayerNorm(latent_dim)
        self.memory     = MemoryModule(NUM_SLOTS, latent_dim, SHRINK_THR)
        self.classifier = nn.Linear(latent_dim, 2)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(latent_dim, 64, kernel_size=1),
            nn.GroupNorm(1, 64), nn.GELU(),
            nn.ConvTranspose1d(64, input_dim, kernel_size=1),
        )

    def forward(self, x):
        z           = self.ln(self.encoder(x.unsqueeze(-1)).squeeze(-1))
        z_hat, _    = self.memory(z)
        recon       = self.decoder(z_hat.unsqueeze(-1)).squeeze(-1)
        return self.classifier(z_hat), recon, z_hat


def build_model(arch: str, input_dim: int) -> nn.Module:
    if arch == 'kan':
        return KanAEClassifier(input_dim, LATENT_DIM)
    elif arch == 'cnn':
        return CnnAEClassifier(input_dim, LATENT_DIM)
    else:
        raise ValueError(f"Unknown ARCH={arch!r}. Choose 'kan' or 'cnn'")


def get_trainable_params(model: nn.Module):
    """
    Return parameters for layer-selective CTTA updates.

    Updated:
      - All LayerNorm / GroupNorm parameters (normalisation statistics)
      - Classifier head (decision boundary can shift toward target)
      - Last encoder layer (final feature representation can adapt)

    Frozen:
      - All early encoder layers (source feature extractors)
      - Decoder (not needed for classification)

    This gives CE a direct path to move the decision boundary while
    protecting source representations from catastrophic forgetting.
    """
    params = []
    seen   = set()

    def add(p):
        if id(p) not in seen:
            seen.add(id(p))
            params.append(p)

    # 1. Norm layers
    for module in model.modules():
        if isinstance(module, (nn.LayerNorm, nn.GroupNorm)):
            for p in module.parameters():
                add(p)

    # 2. Classifier head
    for p in model.classifier.parameters():
        add(p)

    # 3. Last encoder layer
    # KAN: model.encoder is a KAN object with .layers list
    # CNN: model.encoder is nn.Sequential
    encoder = model.encoder
    if isinstance(encoder, nn.Sequential):
        # Last Conv1d in the sequential
        last_layer = None
        for m in encoder.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                last_layer = m
        if last_layer is not None:
            for p in last_layer.parameters():
                add(p)
    else:
        # KAN encoder — last KANLayer
        if hasattr(encoder, 'layers') and len(encoder.layers) > 0:
            for p in encoder.layers[-1].parameters():
                add(p)

    return params


## 4. Training & Evaluation

In [ ]:
def pretrain_source(model, loader, epochs, device):
    """
    Joint supervised + reconstruction pre-training.
    Loss = CrossEntropy(logits, y) + RECON_W * MSE(recon, x)
    ALL parameters updated — standard supervised training.
    """
    optimizer = optim.Adam(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
    ce_crit   = nn.CrossEntropyLoss()
    mse_crit  = nn.MSELoss()
    model.train()

    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits, recon, _ = model(x)
            loss = ce_crit(logits, y) + RECON_W * mse_crit(recon, x)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
        print(f'[Pretrain] Epoch {epoch+1}/{epochs} | '
              f'Loss: {total_loss/len(loader):.4f} | '
              f'Acc: {correct/total:.4f}')


def evaluate(model, loader, device, desc='Eval'):
    """Evaluate using classifier head — argmax of logits."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            logits, _, _ = model(x.to(device))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    f1  = f1_score(labels, preds, zero_division=0)
    acc = accuracy_score(labels, preds)
    print(f'[{desc}] F1: {f1:.4f} | Acc: {acc:.4f}')
    return f1


def run_ctta(model, stream_loader, pool_loader, device):
    """
    Few-Shot Norm-Only CTTA.

    FROZEN:  encoder, classifier head, decoder weights.
    UPDATED: LayerNorm and GroupNorm parameters only.

    Per stream batch (TTA_STEPS gradient steps):
      1. Supervised CE on pool batch — anchors decision boundary to target
      2. Entropy minimisation on stream batch — increases prediction confidence
      3. Reconstruction on pool batch — keeps norm stats grounded in benign

    Returns preds, labels on the stream.
    """
    norm_params = get_trainable_params(model)
    if not norm_params:
        print('[CTTA] WARNING: No trainable parameters found.')
        return evaluate(model, stream_loader, device, desc='CTTA (fallback)')

    print(f'[CTTA] Updating {len(norm_params)} param tensors '
          f'({sum(p.numel() for p in norm_params)} params). '
          f'All other weights frozen.')

    optimizer = optim.Adam(norm_params, lr=TTA_LR)
    ce_crit   = nn.CrossEntropyLoss()
    model.train()

    # Pool cycles indefinitely
    pool_iter = iter(pool_loader)
    def next_pool():
        nonlocal pool_iter
        try:
            return next(pool_iter)
        except StopIteration:
            pool_iter = iter(pool_loader)
            return next(pool_iter)

    all_preds, all_labels = [], []

    for x_stream, y_stream in stream_loader:
        x_stream = x_stream.to(device)

        for _ in range(TTA_STEPS):
            optimizer.zero_grad()

            # ── 1. Supervised CE on pool batch ────────────────────────────
            x_pool, y_pool = next_pool()
            x_pool = x_pool.to(device)
            y_pool = y_pool.to(device)
            logits_pool, recon_pool, _ = model(x_pool)
            loss_ce    = ce_crit(logits_pool, y_pool)

            # ── 2. Entropy on stream batch ────────────────────────────────
            logits_s, _, _ = model(x_stream)
            probs    = F.softmax(logits_s, dim=1)
            loss_ent = -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean()

            # ── 3. Reconstruction on pool batch ───────────────────────────
            loss_recon = F.mse_loss(recon_pool, x_pool)

            loss = (FEW_SHOT_W  * loss_ce   +
                    ENTROPY_W   * loss_ent  +
                    RECON_W_TTA * loss_recon)

            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(norm_params, max_norm=1.0)
                optimizer.step()

        # Final inference
        with torch.no_grad():
            model.eval()
            logits_f, _, _ = model(x_stream)
            preds = logits_f.argmax(1).cpu().numpy()
            model.train()

        all_preds.extend(preds)
        all_labels.extend(y_stream.numpy())

    print('[CTTA] Stream complete.')
    return np.array(all_preds), np.array(all_labels)


## 5. Load Datasets

In [ ]:
# ── Source: ToN-IoT ───────────────────────────────────────────────────────────
X_src, y_src = load_dataset('seyhed/nf-ton-iot-v3', sample_n=SAMPLE_N)
input_dim = X_src.shape[1]
loader_src_train, loader_src_test, scaler = make_source_loaders(X_src, y_src)
del X_src, y_src; gc.collect()

# ── Target 1: UNSW-NB15 ───────────────────────────────────────────────────────
# Fresh MinMaxScaler per dataset — corrects cross-network scale mismatch.
# Benign pool: 1% of target, known-clean only (no attack labels needed).
X_tgt1, y_tgt1 = load_dataset('seyhed/nf-unsw-nb15-v3', sample_n=SAMPLE_N)
pool_tgt1, stream_tgt1 = make_target_loaders(X_tgt1, y_tgt1)
del X_tgt1, y_tgt1; gc.collect()

# ── Target 2: CICIDS2018 ──────────────────────────────────────────────────────
X_tgt2, y_tgt2 = load_dataset('seyhed/nf-cicids2018-v3', sample_n=SAMPLE_N)
pool_tgt2, stream_tgt2 = make_target_loaders(X_tgt2, y_tgt2)
del X_tgt2, y_tgt2; gc.collect()

print(f'\n[System] All datasets loaded. Input dim: {input_dim}')

[Data] Loading seyhed/nf-ton-iot-v3 ...
Using Colab cache for faster access to the 'nf-ton-iot-v3' dataset.
   -> Shape: (100000, 49) | Attack rate: 38.94%
[Data] Loading seyhed/nf-unsw-nb15-v3 ...
Using Colab cache for faster access to the 'nf-unsw-nb15-v3' dataset.
   -> Shape: (100000, 49) | Attack rate: 5.52%
   -> Pool: 1000 samples (attack rate: 5.50%) | Stream: 99000 (attack rate: 5.52%)
[Data] Loading seyhed/nf-cicids2018-v3 ...
Using Colab cache for faster access to the 'nf-cicids2018-v3' dataset.
   -> Shape: (100000, 49) | Attack rate: 12.95%
   -> Pool: 1000 samples (attack rate: 12.90%) | Stream: 99000 (attack rate: 12.95%)

[System] All datasets loaded. Input dim: 49


## 6. Phase 1 — Source Pre-training

In [ ]:
print('=' * 60)
print(f'PHASE 1: SOURCE PRE-TRAINING (ToN-IoT) | {ARCH.upper()}')
print('=' * 60)

model = build_model(ARCH, input_dim).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_params = get_trainable_params(model)
print(f'[Model] {ARCH} | input_dim={input_dim} | latent_dim={LATENT_DIM} | '
      f'total params={n_params:,} | trainable params={sum(p.numel() for p in trainable_params)}\n')

pretrain_source(model, loader_src_train, epochs=PRETRAIN_EPOCHS, device=device)

PHASE 1: SOURCE PRE-TRAINING (ToN-IoT) | KAN
[Model] kan | input_dim=49 | latent_dim=32 | total params=107,010 | trainable params=20610

[Pretrain] Epoch 1/20 | Loss: 0.4224 | Acc: 0.8434
[Pretrain] Epoch 2/20 | Loss: 0.2675 | Acc: 0.8964
[Pretrain] Epoch 3/20 | Loss: 0.2379 | Acc: 0.9076
[Pretrain] Epoch 4/20 | Loss: 0.2277 | Acc: 0.9160
[Pretrain] Epoch 5/20 | Loss: 0.2248 | Acc: 0.9184
[Pretrain] Epoch 6/20 | Loss: 0.2171 | Acc: 0.9230
[Pretrain] Epoch 7/20 | Loss: 0.2129 | Acc: 0.9237
[Pretrain] Epoch 8/20 | Loss: 0.2107 | Acc: 0.9246
[Pretrain] Epoch 9/20 | Loss: 0.2097 | Acc: 0.9246
[Pretrain] Epoch 10/20 | Loss: 0.2085 | Acc: 0.9261
[Pretrain] Epoch 11/20 | Loss: 0.2065 | Acc: 0.9268
[Pretrain] Epoch 12/20 | Loss: 0.2048 | Acc: 0.9284
[Pretrain] Epoch 13/20 | Loss: 0.2042 | Acc: 0.9292
[Pretrain] Epoch 14/20 | Loss: 0.2023 | Acc: 0.9313
[Pretrain] Epoch 15/20 | Loss: 0.2028 | Acc: 0.9302
[Pretrain] Epoch 16/20 | Loss: 0.2024 | Acc: 0.9310
[Pretrain] Epoch 17/20 | Loss: 0.2008 | 

## 7. Diagnostic — Post-Pretraining

In [ ]:
# Source F1 should be >0.85 before CTTA.
# Zero-shot gives the baseline CTTA should improve from.
print('[Diagnostic] Source test performance:')
evaluate(model, loader_src_test, device, desc='Source (ToN-IoT) [post-pretrain]')

print('\n[Diagnostic] Zero-shot on targets (no adaptation yet):')
evaluate(model, stream_tgt1, device, desc='Target1 (UNSW-NB15)  [zero-shot]')
evaluate(model, stream_tgt2, device, desc='Target2 (CICIDS2018) [zero-shot]')

print('\n[Diagnostic] Parameters that will be updated during CTTA:')
total_norm = 0
for name, module in model.named_modules():
    if isinstance(module, (nn.LayerNorm, nn.GroupNorm, nn.Linear, nn.Conv1d)):
        n = sum(p.numel() for p in module.parameters())
        total_norm += n
        print(f'  {name:40s} ({module.__class__.__name__}, {n} params)')
total_all = sum(p.numel() for p in model.parameters())
print(f'\n  Updating {total_norm} / {total_all} params '
      f'({total_norm/total_all:.2%} of model) during CTTA.')


[Diagnostic] Source test performance:
[Source (ToN-IoT) [post-pretrain]] F1: 0.9064 | Acc: 0.9264

[Diagnostic] Zero-shot on targets (no adaptation yet):
[Target1 (UNSW-NB15)  [zero-shot]] F1: 0.0550 | Acc: 0.8267
[Target2 (CICIDS2018) [zero-shot]] F1: 0.2017 | Acc: 0.6355

[Diagnostic] Parameters that will be updated during CTTA:
  ln                                       (LayerNorm, 64 params)
  classifier                               (Linear, 66 params)

  Updating 130 / 107010 params (0.12% of model) during CTTA.


## 8. Phase 2 — Zero-Shot Baseline

In [ ]:
print('=' * 60)
print('PHASE 2: ZERO-SHOT CROSS-DOMAIN EVALUATION')
print('=' * 60)

evaluate(model, loader_src_test, device, desc='Source  (ToN-IoT)   [zero-shot]')
evaluate(model, stream_tgt1,     device, desc='Target1 (UNSW-NB15) [zero-shot]')
evaluate(model, stream_tgt2,     device, desc='Target2 (CICIDS2018)[zero-shot]')

PHASE 2: ZERO-SHOT CROSS-DOMAIN EVALUATION
[Source  (ToN-IoT)   [zero-shot]] F1: 0.9064 | Acc: 0.9264
[Target1 (UNSW-NB15) [zero-shot]] F1: 0.0550 | Acc: 0.8267
[Target2 (CICIDS2018)[zero-shot]] F1: 0.2017 | Acc: 0.6355


0.2017078134678347

## 9. Phase 3 — CTTA on Target 1 (UNSW-NB15)

Only norm layer parameters are updated. All other weights are frozen.
The decision boundary is physically preserved.

In [ ]:
print('=' * 60)
print('PHASE 3: NORM-ONLY CTTA — Target 1 (UNSW-NB15)')
print('=' * 60)

# Reset norm parameters to post-pretrain state before each CTTA phase
# so phases are independent and comparable
torch.manual_seed(SEED)
model_state = {k: v.clone() for k, v in model.state_dict().items()}

preds_tgt1, labels_tgt1 = run_ctta(
    model,
    stream_loader = stream_tgt1,
    pool_loader   = pool_tgt1,
    device        = device,
)
f1  = f1_score(labels_tgt1, preds_tgt1, zero_division=0)
acc = accuracy_score(labels_tgt1, preds_tgt1)
print(f'[CTTA Target1] F1: {f1:.4f} | Acc: {acc:.4f}')

PHASE 3: NORM-ONLY CTTA — Target 1 (UNSW-NB15)
[CTTA] Updating 7 param tensors (20610 params). All other weights frozen.
[CTTA] Stream complete.
[CTTA Target1] F1: 0.7429 | Acc: 0.9768


## 10. Phase 4 — CTTA on Target 2 (CICIDS2018)

Model is reset to post-pretrain state before this phase
so results are independent of Phase 3.

In [ ]:
print('=' * 60)
print('PHASE 4: NORM-ONLY CTTA — Target 2 (CICIDS2018)')
print('=' * 60)

# Reset to post-pretrain state
model.load_state_dict(model_state)

preds_tgt2, labels_tgt2 = run_ctta(
    model,
    stream_loader = stream_tgt2,
    pool_loader   = pool_tgt2,
    device        = device,
)
f1  = f1_score(labels_tgt2, preds_tgt2, zero_division=0)
acc = accuracy_score(labels_tgt2, preds_tgt2)
print(f'[CTTA Target2] F1: {f1:.4f} | Acc: {acc:.4f}')

PHASE 4: NORM-ONLY CTTA — Target 2 (CICIDS2018)
[CTTA] Updating 7 param tensors (20610 params). All other weights frozen.
[CTTA] Stream complete.
[CTTA Target2] F1: 0.4556 | Acc: 0.8903


## 11. Retention Check

Reset model to post-pretrain state then evaluate source.
Since only norm params were updated during CTTA, retention
should be close to 100%.

In [ ]:
print('=' * 60)
print('RETENTION CHECK: Source (ToN-IoT) After CTTA')
print('=' * 60)

# Restore to post-pretrain state
model.load_state_dict(model_state)
evaluate(model, loader_src_test, device, desc='Source (ToN-IoT) [post-CTTA state]')

print('\n[Note] Source retention should be ~100% because only norm params')
print('were updated — the encoder and classifier weights are unchanged.')

RETENTION CHECK: Source (ToN-IoT) After CTTA
[Source (ToN-IoT) [post-CTTA state]] F1: 0.9064 | Acc: 0.9264

[Note] Source retention should be ~100% because only norm params
were updated — the encoder and classifier weights are unchanged.
